# FinQA benchmark walkthrough

This notebook runs `BenchmarkRunner` against the FinQA dataset using a small HuggingFace model, plots a reliability diagram, and saves a `BenchmarkResult` JSON record that can later be rendered into an AI RMF report.

> **Runtime note.** This notebook downloads `Qwen/Qwen2.5-0.5B` (~1 GB) from HuggingFace on first run and performs 50 real model generations. On a CPU this can take 5–15 minutes; on a GPU it is near-instant. If you only want to verify the code path, switch `backend="hf"` to `backend="dummy"` and `model="Qwen/Qwen2.5-0.5B"` to `model="dummy-0"` in the next cell.

## 1. Build the pipeline

We use Qwen2.5-0.5B — small enough to run on a CPU in a pinch, token-logprob-capable, and Apache-licensed. Swap in any other HF causal LM you prefer.

In [ ]:
from lub.pipeline import UncertaintyPipeline

pipe = UncertaintyPipeline.from_pretrained(
    model="Qwen/Qwen2.5-0.5B",
    backend="hf",
    estimator="token_logprob",
    refusal_threshold=0.3,
)

## 2. Run the benchmark

`limit=50` keeps the cell fast; scale up once you have a stable baseline. `seed=0` makes the resulting `BenchmarkResult` deterministic.

In [ ]:
from lub.benchmarks import BenchmarkRunner, FinQADataset

runner = BenchmarkRunner(pipeline=pipe, dataset=FinQADataset(split="test"))
result = runner.run(limit=50, seed=0)
print(result.model_dump_json(indent=2))

## 3. Reliability diagram

Expected Calibration Error is a single scalar; the reliability diagram is where you actually *see* where the model is over- or under-confident. Overlay the diagonal and look for systematic bias in the high-confidence buckets.

In [ ]:
import json
from pathlib import Path

from lub.calibration.plots import plot_reliability_diagram

per_example = Path("finqa_per_example.json")
records = json.loads(per_example.read_text()) if per_example.exists() else []
confs = [r["confidence"] for r in records]
correct = [r["correct"] for r in records]
fig = plot_reliability_diagram(confs, correct, title="FinQA (Qwen2.5-0.5B)")
fig

## 4. Persist the result

Write the `BenchmarkResult` JSON next to the notebook. Notebook 03 reads it back to render an AI RMF report.

In [ ]:
Path("finqa_result.json").write_text(result.model_dump_json(indent=2))

## Discussion

A small model evaluated with token-logprob confidence is expected to be over-confident on FinQA: the estimator sees long, low-entropy generations and hands back high `exp(mean(logprob))` values even when the numerical answer is wrong. The ECE number will usually exceed 0.1, and the refusal AUROC will sit well below a semantic-entropy baseline. Treat this notebook as the *starting* point for a calibration investigation, not a certification — swap estimators, add a conformal calibration set, and re-run.